# ICASN

In [1]:
# !/usr/bin/env python3

import pandas as pd
import numpy as np
import ROOT
import sys
import os

# ------------------ CHECK FILES ------------------
required_files = [
    "AtlasStyle.C",
    "iitm_MALTA2testing-threshold_noise_irrediated.csv",
    "iitm_MALTA2testing-threshold_noise_unirrediated.csv"
]

for f in required_files:
    if not os.path.exists(f):
        print(f"ERROR: Missing file -> {f}")
        sys.exit(1)

# ------------------ LOAD ATLAS STYLE ------------------
ROOT.gROOT.SetBatch(True)
ROOT.gROOT.LoadMacro("AtlasStyle.C")
ROOT.SetAtlasStyle()
ROOT.gStyle.SetPaintTextFormat("4.0f")

# ------------------ FILE PATHS ------------------
file_irrad = "iitm_MALTA2testing-threshold_noise_irrediated.csv"
file_unirrad = "iitm_MALTA2testing-threshold_noise_unirrediated.csv"

# ------------------ LOAD DATA ------------------
df_irrad = pd.read_csv(file_irrad, na_values=['m',' ','Nan','nan'])
df_unirrad = pd.read_csv(file_unirrad, na_values=['m',' ','Nan','nan'])

cols = ["ITHR","IDB","ICASN","TH_MEAN","TH_SD"]

df_irrad = df_irrad[cols].dropna(subset=["TH_MEAN"])
df_unirrad = df_unirrad[cols].dropna(subset=["TH_MEAN"])

df_unirrad["ITHR"] = pd.to_numeric(df_unirrad["ITHR"], errors='coerce')
df_unirrad["IDB"] = pd.to_numeric(df_unirrad["IDB"], errors='coerce')

df_unirrad = df_unirrad.dropna(subset=["ITHR","IDB"])

# ------------------ FUNCTIONS ------------------

def make_graph(x, y, yerr):
    g = ROOT.TGraphErrors(len(x))
    for i in range(len(x)):
        g.SetPoint(i, float(x[i]), float(y[i]))
        g.SetPointError(i, 0, float(yerr[i]))
    return g


def make_graph_no_error(x, y):
    g = ROOT.TGraph(len(x))
    for i in range(len(x)):
        g.SetPoint(i, float(x[i]), float(y[i]))
    return g


# ------------------ OUTPUT DIR ------------------
os.makedirs("plots", exist_ok=True)

# ------------------ GRAPH SECTION ------------------

common_icasn = sorted(set(df_irrad["ICASN"]) & set(df_unirrad["ICASN"]))

mean_ir = df_irrad.groupby("ICASN")["TH_MEAN"].mean().reindex(common_icasn)
mean_un = df_unirrad.groupby("ICASN")["TH_MEAN"].mean().reindex(common_icasn)

std_ir = df_irrad.groupby("ICASN")["TH_SD"].mean().reindex(common_icasn)
std_un = df_unirrad.groupby("ICASN")["TH_SD"].mean().reindex(common_icasn)

mask = (~mean_ir.isna()) & (~mean_un.isna())

x_vals = np.array(common_icasn)[mask]
y_ir = mean_ir[mask].values
y_un = mean_un[mask].values
e_ir = std_ir[mask].values
e_un = std_un[mask].values

# ------------------ MAIN 1D GRAPH ------------------

g1 = make_graph(x_vals, y_un, e_un)
g2 = make_graph(x_vals, y_ir, e_ir)

# Styling
g1.SetMarkerStyle(20)
g2.SetMarkerStyle(21)

g1.SetMarkerColor(ROOT.kBlue)
g2.SetMarkerColor(ROOT.kRed)

g1.SetLineColor(ROOT.kBlue)
g2.SetLineColor(ROOT.kRed)

g1.SetLineWidth(2)
g2.SetLineWidth(2)

g1.SetMarkerSize(1.2)
g2.SetMarkerSize(1.2)

# Canvas
c2 = ROOT.TCanvas("c2", "", 800, 600)

g1.Draw("APL")
g1.GetXaxis().SetTitle("ICASN")
g1.GetYaxis().SetTitle("Mean Threshold (e^{-})")
g1.GetYaxis().SetRangeUser(400, 700)

g2.Draw("PL SAME")

# Legend
leg = ROOT.TLegend(0.65,0.75,0.88,0.88)
leg.AddEntry(g1, "Unirradiated", "lp")
leg.AddEntry(g2, "Irradiated", "lp")
leg.SetBorderSize(0)
leg.Draw()

ROOT.gPad.SetTicks(1,1)

c2.SaveAs("plots/Threshold_vs_ICASN.pdf")

# ------------------ DELTA PLOT ------------------

delta = mean_un - mean_ir
delta = delta.reindex(common_icasn)

mask_delta = ~delta.isna()

x_delta = np.array(common_icasn)[mask_delta]
y_delta = delta[mask_delta].values

g_delta = make_graph_no_error(x_delta, y_delta)

# Styling
g_delta.SetMarkerStyle(20)
g_delta.SetMarkerColor(ROOT.kBlack)
g_delta.SetLineColor(ROOT.kBlack)
g_delta.SetLineWidth(2)
g_delta.SetMarkerSize(1.2)

# Canvas
c3 = ROOT.TCanvas("c3", "", 800, 600)

g_delta.Draw("APL")
g_delta.GetXaxis().SetTitle("ICASN")
g_delta.GetYaxis().SetTitle("Threshold Shift (e^{-})")
g_delta.SetTitle("Radiation Damage vs ICASN")

ROOT.gPad.SetTicks(1,1)

c3.SaveAs("plots/Threshold_shift_vs_ICASN.pdf")

print("\n✅ All plots saved in ./plots/")


Applying ATLAS style settings...



Info in <TCanvas::Print>: pdf file plots/Threshold_vs_ICASN.pdf has been created
Info in <TCanvas::Print>: pdf file plots/Threshold_shift_vs_ICASN.pdf has been created



✅ All plots saved in ./plots/


# IDB

In [2]:
#!/usr/bin/env python3

import pandas as pd
import numpy as np
import ROOT
import sys
import os

# ------------------ CHECK FILES ------------------
required_files = [
    "AtlasStyle.C",
    "iitm_MALTA2testing-threshold_noise_irrediated.csv",
    "iitm_MALTA2testing-threshold_noise_unirrediated.csv"
]

for f in required_files:
    if not os.path.exists(f):
        print(f"ERROR: Missing file -> {f}")
        sys.exit(1)

# ------------------ LOAD ATLAS STYLE ------------------
ROOT.gROOT.SetBatch(True)
ROOT.gROOT.LoadMacro("AtlasStyle.C")
ROOT.SetAtlasStyle()
ROOT.gStyle.SetPaintTextFormat("4.0f")

# ------------------ FILE PATHS ------------------
file_irrad = "iitm_MALTA2testing-threshold_noise_irrediated.csv"
file_unirrad = "iitm_MALTA2testing-threshold_noise_unirrediated.csv"

# ------------------ LOAD DATA ------------------
df_irrad = pd.read_csv(file_irrad, na_values=['m',' ','Nan','nan'])
df_unirrad = pd.read_csv(file_unirrad, na_values=['m',' ','Nan','nan'])

cols = ["ITHR","IDB","ICASN","TH_MEAN","TH_SD"]

df_irrad = df_irrad[cols].dropna(subset=["TH_MEAN"])
df_unirrad = df_unirrad[cols].dropna(subset=["TH_MEAN"])

df_unirrad["ITHR"] = pd.to_numeric(df_unirrad["ITHR"], errors='coerce')
df_unirrad["IDB"] = pd.to_numeric(df_unirrad["IDB"], errors='coerce')

df_unirrad = df_unirrad.dropna(subset=["ITHR","IDB"])

# ------------------ FUNCTIONS ------------------

def make_graph(x, y, yerr):
    g = ROOT.TGraphErrors(len(x))
    for i in range(len(x)):
        g.SetPoint(i, float(x[i]), float(y[i]))
        g.SetPointError(i, 0, float(yerr[i]))
    return g


def make_graph_no_error(x, y):
    g = ROOT.TGraph(len(x))
    for i in range(len(x)):
        g.SetPoint(i, float(x[i]), float(y[i]))
    return g


# ------------------ OUTPUT DIR ------------------
os.makedirs("plots", exist_ok=True)

# ------------------ GRAPH SECTION ------------------

common_idb = sorted(set(df_irrad["IDB"]) & set(df_unirrad["IDB"]))

mean_ir = df_irrad.groupby("IDB")["TH_MEAN"].mean().reindex(common_idb)
mean_un = df_unirrad.groupby("IDB")["TH_MEAN"].mean().reindex(common_idb)

std_ir = df_irrad.groupby("IDB")["TH_SD"].mean().reindex(common_idb)
std_un = df_unirrad.groupby("IDB")["TH_SD"].mean().reindex(common_idb)

mask = (~mean_ir.isna()) & (~mean_un.isna())

x_vals = np.array(common_idb)[mask]
y_ir = mean_ir[mask].values
y_un = mean_un[mask].values
e_ir = std_ir[mask].values
e_un = std_un[mask].values

# ------------------ MAIN 1D GRAPH ------------------

g1 = make_graph(x_vals, y_un, e_un)
g2 = make_graph(x_vals, y_ir, e_ir)

# Styling
g1.SetMarkerStyle(20)
g2.SetMarkerStyle(21)

g1.SetMarkerColor(ROOT.kBlue)
g2.SetMarkerColor(ROOT.kRed)

g1.SetLineColor(ROOT.kBlue)
g2.SetLineColor(ROOT.kRed)

g1.SetLineWidth(2)
g2.SetLineWidth(2)

g1.SetMarkerSize(1.2)
g2.SetMarkerSize(1.2)

# Canvas
c2 = ROOT.TCanvas("c2", "", 800, 600)

g1.Draw("APL")
g1.GetXaxis().SetTitle("IDB")
g1.GetYaxis().SetTitle("Mean Threshold (e^{-})")
g1.GetYaxis().SetRangeUser(400, 700)

g2.Draw("PL SAME")

# Legend
leg = ROOT.TLegend(0.65,0.75,0.88,0.88)
leg.AddEntry(g1, "Unirradiated", "lp")
leg.AddEntry(g2, "Irradiated", "lp")
leg.SetBorderSize(0)
leg.Draw()

ROOT.gPad.SetTicks(1,1)

c2.SaveAs("plots/Threshold_vs_IDB.pdf")

# ------------------ DELTA PLOT ------------------

delta = mean_un - mean_ir
delta = delta.reindex(common_idb)

mask_delta = ~delta.isna()

x_delta = np.array(common_idb)[mask_delta]
y_delta = delta[mask_delta].values

g_delta = make_graph_no_error(x_delta, y_delta)

# Styling
g_delta.SetMarkerStyle(20)
g_delta.SetMarkerColor(ROOT.kBlack)
g_delta.SetLineColor(ROOT.kBlack)
g_delta.SetLineWidth(2)
g_delta.SetMarkerSize(1.2)

# Canvas
c3 = ROOT.TCanvas("c3", "", 800, 600)

g_delta.Draw("APL")
g_delta.GetXaxis().SetTitle("IDB")
g_delta.GetYaxis().SetTitle("Threshold Shift (e^{-})")
g_delta.SetTitle("Radiation Damage vs IDB")

ROOT.gPad.SetTicks(1,1)

c3.SaveAs("plots/Threshold_shift_vs_IDB.pdf")

print("\n✅ All IDB plots saved in ./plots/")


✅ All IDB plots saved in ./plots/

Applying ATLAS style settings...



Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: pdf file plots/Threshold_vs_IDB.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: pdf file plots/Threshold_shift_vs_IDB.pdf has been created


# ITHR

In [3]:
#!/usr/bin/env python3

import pandas as pd
import numpy as np
import ROOT
import sys
import os

# ------------------ CHECK FILES ------------------
required_files = [
    "AtlasStyle.C",
    "iitm_MALTA2testing-threshold_noise_irrediated.csv",
    "iitm_MALTA2testing-threshold_noise_unirrediated.csv"
]

for f in required_files:
    if not os.path.exists(f):
        print(f"ERROR: Missing file -> {f}")
        sys.exit(1)

# ------------------ LOAD ATLAS STYLE ------------------
ROOT.gROOT.SetBatch(True)
ROOT.gROOT.LoadMacro("AtlasStyle.C")
ROOT.SetAtlasStyle()
ROOT.gStyle.SetPaintTextFormat("4.0f")

# ------------------ FILE PATHS ------------------
file_irrad = "iitm_MALTA2testing-threshold_noise_irrediated.csv"
file_unirrad = "iitm_MALTA2testing-threshold_noise_unirrediated.csv"

# ------------------ LOAD DATA ------------------
df_irrad = pd.read_csv(file_irrad, na_values=['m',' ','Nan','nan'])
df_unirrad = pd.read_csv(file_unirrad, na_values=['m',' ','Nan','nan'])

cols = ["ITHR","IDB","ICASN","TH_MEAN","TH_SD"]

df_irrad = df_irrad[cols].dropna(subset=["TH_MEAN"])
df_unirrad = df_unirrad[cols].dropna(subset=["TH_MEAN"])

df_unirrad["ITHR"] = pd.to_numeric(df_unirrad["ITHR"], errors='coerce')
df_unirrad["IDB"] = pd.to_numeric(df_unirrad["IDB"], errors='coerce')

df_unirrad = df_unirrad.dropna(subset=["ITHR","IDB"])

# ------------------ FUNCTIONS ------------------

def make_graph(x, y, yerr):
    g = ROOT.TGraphErrors(len(x))
    for i in range(len(x)):
        g.SetPoint(i, float(x[i]), float(y[i]))
        g.SetPointError(i, 0, float(yerr[i]))
    return g


def make_graph_no_error(x, y):
    g = ROOT.TGraph(len(x))
    for i in range(len(x)):
        g.SetPoint(i, float(x[i]), float(y[i]))
    return g


# ------------------ OUTPUT DIR ------------------
os.makedirs("plots", exist_ok=True)

# ------------------ GRAPH SECTION ------------------

common_ithr = sorted(set(df_irrad["ITHR"]) & set(df_unirrad["ITHR"]))

mean_ir = df_irrad.groupby("ITHR")["TH_MEAN"].mean().reindex(common_ithr)
mean_un = df_unirrad.groupby("ITHR")["TH_MEAN"].mean().reindex(common_ithr)

std_ir = df_irrad.groupby("ITHR")["TH_SD"].mean().reindex(common_ithr)
std_un = df_unirrad.groupby("ITHR")["TH_SD"].mean().reindex(common_ithr)

mask = (~mean_ir.isna()) & (~mean_un.isna())

x_vals = np.array(common_ithr)[mask]
y_ir = mean_ir[mask].values
y_un = mean_un[mask].values
e_ir = std_ir[mask].values
e_un = std_un[mask].values

# ------------------ MAIN 1D GRAPH ------------------

g1 = make_graph(x_vals, y_un, e_un)
g2 = make_graph(x_vals, y_ir, e_ir)

# Styling
g1.SetMarkerStyle(20)
g2.SetMarkerStyle(21)

g1.SetMarkerColor(ROOT.kBlue)
g2.SetMarkerColor(ROOT.kRed)

g1.SetLineColor(ROOT.kBlue)
g2.SetLineColor(ROOT.kRed)

g1.SetLineWidth(2)
g2.SetLineWidth(2)

g1.SetMarkerSize(1.2)
g2.SetMarkerSize(1.2)

# Canvas
c2 = ROOT.TCanvas("c2", "", 800, 600)

g1.Draw("APL")
g1.GetXaxis().SetTitle("ITHR")
g1.GetYaxis().SetTitle("Mean Threshold (e^{-})")
g1.GetYaxis().SetRangeUser(200, 1000)

g2.Draw("PL SAME")

# Legend
leg = ROOT.TLegend(0.65,0.75,0.88,0.88)
leg.AddEntry(g1, "Unirradiated", "lp")
leg.AddEntry(g2, "Irradiated", "lp")
leg.SetBorderSize(0)
leg.Draw()

ROOT.gPad.SetTicks(1,1)

c2.SaveAs("plots/Threshold_vs_ITHR.pdf")

# ------------------ DELTA PLOT ------------------

delta = mean_un - mean_ir
delta = delta.reindex(common_ithr)

mask_delta = ~delta.isna()

x_delta = np.array(common_ithr)[mask_delta]
y_delta = delta[mask_delta].values

g_delta = make_graph_no_error(x_delta, y_delta)

# Styling
g_delta.SetMarkerStyle(20)
g_delta.SetMarkerColor(ROOT.kBlack)
g_delta.SetLineColor(ROOT.kBlack)
g_delta.SetLineWidth(2)
g_delta.SetMarkerSize(1.2)

# Canvas
c3 = ROOT.TCanvas("c3", "", 800, 600)

g_delta.Draw("APL")
g_delta.GetXaxis().SetTitle("ITHR")
g_delta.GetYaxis().SetTitle("Threshold Shift (e^{-})")
g_delta.SetTitle("Radiation Damage vs ITHR")

ROOT.gPad.SetTicks(1,1)

c3.SaveAs("plots/Threshold_shift_vs_ITHR.pdf")

print("\n✅ All ITHR plots saved in ./plots/")


✅ All ITHR plots saved in ./plots/

Applying ATLAS style settings...



Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: pdf file plots/Threshold_vs_ITHR.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: pdf file plots/Threshold_shift_vs_ITHR.pdf has been created
